# Spark: bank-loan data analysis

> **Pilot status:** run this notebook only after the organiser has validated the
> prepared Python/PySpark/Java environment, dataset access and Spark networking.
> Do not install software or manage Docker containers yourself.

## Learning objectives

By the end of this tutorial, you should be able to:

- Explain the roles of a Spark driver, master, workers and executors.
- Load a CSV into a Spark DataFrame and inspect its schema.
- Select columns, filter records and aggregate category counts.
- Distinguish transformations from actions that trigger execution.
- Export a bounded result to your own workspace and stop your Spark session.

## How to work

Open this notebook from the Spark workspace directory in JupyterLab. Select the
organiser-prepared Python kernel, then run the cells in order using **Shift+Enter**.
Later cells depend on variables created by earlier cells. An execution indicator
means a cell is still running; wait before proceeding.

Prefer a terminal? `data-analytics.py` is the documented alternative and performs
these same analysis activities. Use the organiser-prepared terminal interpreter;
the terminal's default Python may differ from the notebook kernel. Use one
interface at a time: stop its Spark session before switching, especially when
fixed callback ports are configured. Both interfaces replace the same preview
file under `outputs/`.

The following notes describe expected outcomes, not evidence of a successful live
run. Stored execution results remain unchanged by this documentation update.

## 1. Inspect your local dataset

The CSV is included under `data/`. Preview it with Python's CSV reader before
asking Spark to process it. This activity reads a local file and does not connect
to the cluster.

**Expected outcome:** the cell prints the column names and up to five data
records. If it reports that the file is missing, check that the notebook was
opened from the Spark workspace; otherwise ask the organiser to check delivery.

`Income` and other field meanings, units and category codes must be checked
against the accompanying teaching material and dataset source. The CSV itself
is not a data dictionary. Do not assume that a numeric category is a measurement.

**Data-quality discussion:** look for missing or implausible values. Investigate
such values rather than silently repairing them. This tutorial does not clean
or modify the source dataset.

In [ ]:
import csv
from pathlib import Path

workspace = Path.cwd()
dataset = workspace / 'data' / 'Bank_Personal_Loan_Modelling.csv'
if not dataset.is_file():
    raise FileNotFoundError('Open this notebook from the Spark workspace directory.')

with dataset.open(newline='', encoding='utf-8-sig') as handle:
    reader = csv.DictReader(handle)
    print('Columns:', ', '.join(reader.fieldnames))
    for index, row in enumerate(reader):
        if index == 5:
            break
        print(row)


## 2. Connect to the prepared Spark cluster

### Understand the components

- **Driver:** the Python process running this notebook. It creates the execution
  plan and coordinates the Spark application.
- **Master:** the standalone cluster service that coordinates available worker
  resources and application resource allocation.
- **Workers:** cluster services that launch executors for applications.
- **Executors:** application processes that perform tasks on data partitions.

The driver connects to the master, but executors also need to connect back to the
driver. Being able to reach the master alone is not sufficient for a working job.

### Prepared configuration

The organiser supplies:

- `SPARK_MASTER_URL`: the assigned cluster's Spark master URL.
- `SPARK_DATA_PATH`: the CSV path accessible to the driver and executors.
- `SPARK_DRIVER_HOST`: the driver address reachable from the workers.
- `SPARK_DRIVER_PORT` and `SPARK_BLOCK_MANAGER_PORT`: optional fixed ports when
  required by the prepared network setup.

The local preview path and Spark input path may differ, but must refer to the
same supplied dataset. Do not replace the prepared settings with guessed values.
Binding the driver to local interfaces does not configure routing or firewall
access; those are organiser responsibilities.

**Expected outcome:** the cell creates a Spark session and prints its version
and master URL. This confirms session creation, not yet successful distributed
processing. The data actions in the next sections provide further checks.

### If something goes wrong

| Symptom | Participant action |
|---|---|
| Missing configuration | Stop and report the named missing settings. |
| Missing PySpark or Java / startup error | Confirm the selected kernel with the organiser; do not install dependencies. |
| Timeout, refused connection or unreachable address | Interrupt the blocked cell and report the error; do not change ports or start containers. |
| Input path not found in a later cell | Report the Spark input-path error; the organiser must check driver and executor data access. |

Do not continue dependent cells after a failure. If a usable `spark` session was
created, run the final cleanup cell before restarting or changing interfaces.

In [ ]:
import os

required_settings = ('SPARK_MASTER_URL', 'SPARK_DATA_PATH', 'SPARK_DRIVER_HOST')
missing = [name for name in required_settings if not os.environ.get(name)]
if missing:
    raise RuntimeError('Workspace preparation is incomplete: missing ' + ', '.join(missing))

from pyspark.sql import SparkSession

builder = (
    SparkSession.builder
    .appName('BankLoanAnalysis')
    .master(os.environ['SPARK_MASTER_URL'])
    .config('spark.driver.host', os.environ['SPARK_DRIVER_HOST'])
    .config('spark.driver.bindAddress', '0.0.0.0')
)
for setting, environment in (
    ('spark.driver.port', 'SPARK_DRIVER_PORT'),
    ('spark.blockManager.port', 'SPARK_BLOCK_MANAGER_PORT'),
):
    if os.environ.get(environment):
        builder = builder.config(setting, os.environ[environment])

spark = builder.getOrCreate()
print('Spark version:', spark.version)
print('Spark master:', spark.sparkContext.master)


## 3. Read the CSV and inspect its schema

`header` uses the first CSV row as column names. `inferSchema` asks Spark to infer
column types from the input. These are the same options used by the Python script.
A **DataFrame** represents tabular data with named, typed columns.

The cell prints the schema, displays five rows and counts the full input.
`show` and `count` are actions that trigger processing; schema inference may
also require Spark to inspect the input.

**Expected outcome:** a schema, a five-row preview and **5,000 records** when the
complete supplied dataset is used. If the count differs, report it rather than
editing the CSV to match the expected value.

**Check your understanding:** inspect the inferred types of `Age`, `Income` and
`Family`. Which fields are measurements and which represent categories or counts?
Use the dataset documentation to support your interpretation.

In [ ]:
df = (
    spark.read.format('csv')
    .option('header', 'true')
    .option('inferSchema', 'true')
    .load(os.environ['SPARK_DATA_PATH'])
)
df.printSchema()
df.show(5)
print('Number of records:', df.count())


## 4. Filter customers older than 30

Select `Age`, `Income` and `Family`, then retain records where `Age > 30`.
The condition is strictly greater than 30: records with `Age == 30` are excluded.

`select` and `filter` are **transformations**: they construct an execution plan.
`show` and `count` are **actions**: they request results and trigger execution.
This deferred execution is called *lazy evaluation*.

**Expected outcome:** a preview of up to ten matching rows containing only the
three selected columns, followed by the full number of matching records. The
preview is not the complete filtered dataset. The exact matching count must be
obtained from the supplied data; it is not assumed here.

**Check your understanding:** why can the displayed preview contain fewer rows
than the matching-record count? Do all displayed ages satisfy the strict filter?

In [ ]:
result_df = df.select('Age', 'Income', 'Family').filter(df.Age > 30)
result_df.show(10)
print('Matching records:', result_df.count())


## 5. Explore the loan outcome

Count records for each value of `Personal Loan`. `groupBy` forms groups with the
same category value, and `count` calculates their sizes. `orderBy` sorts the
summary by category value before it is displayed.

**Expected outcome:** a small table with one row per observed category and its
record count. Check the source documentation before assigning meanings to the
category codes. This exercise describes the dataset; it does not train a loan
prediction model or establish a causal relationship.

**Check your understanding:** which category contains more records? The category
counts should sum to the input count from Section 3, including a null category
if one is present.

Spark can distribute aggregation across input partitions. The displayed summary
alone does not establish that every available worker performed a task; execution
placement requires separate observation and validation.

In [ ]:
df.groupBy('Personal Loan').count().orderBy('Personal Loan').show()


## 6. Save a small preview in your own workspace

Collect at most ten filtered rows to the driver, then write one local CSV. This
bounded collection avoids bringing the entire distributed dataset into driver
memory. Do not remove the limit to export an arbitrarily large dataset this way.

**Expected outcome:** the cell prints the path to
`outputs/filtered_preview.csv`. Find it in JupyterLab's file browser. It contains
the `Age`, `Income` and `Family` header and at most ten data rows. An empty filtered
result produces a header-only file.

**Important behaviour:**

- Re-running this cell replaces the existing file, including a preview exported
  by `data-analytics.py`. Download or rename a result first if you need to keep it.
- No explicit ordering is applied, so the selected rows need not be identical
  between runs.
- The file is written locally in your own workspace, not into a shared worker
  dataset directory. The input CSV is not modified.
- This is a single preview CSV, not Spark's distributed CSV output, which normally
  consists of a directory containing part files.

**Check your understanding:** why is collecting ten rows safer than collecting
the entire dataset? What information is lost by exporting only a preview?

In [ ]:
output_dir = workspace / 'outputs'
output_dir.mkdir(exist_ok=True)
output_file = output_dir / 'filtered_preview.csv'

with output_file.open('w', newline='', encoding='utf-8') as handle:
    writer = csv.writer(handle)
    writer.writerow(result_df.columns)
    for row in result_df.limit(10).collect():
        writer.writerow(tuple(row))

print('Saved:', output_file)


## 7. Finish

Run the final cell when you have finished. It calls `spark.stop()` to release this
application's Spark session; it does not stop the organiser's cluster containers.
The notebook does not automatically run this cell after an earlier failure.

**Expected outcome:** `Spark session stopped.` To repeat the Spark exercises,
restart the notebook kernel and run the notebook from the beginning. Do not use
DataFrames belonging to a stopped session.

### Completion checklist

- [ ] I inspected the local CSV and explained its relevant fields using the source.
- [ ] I identified the driver, master, workers and executors.
- [ ] I inspected the schema and checked the complete input count.
- [ ] I explained the strict age filter and distinguished preview size from count.
- [ ] I interpreted the category summary using verified category definitions.
- [ ] I found the bounded preview CSV and understood its overwrite behaviour.
- [ ] I stopped my Spark session.

For additional background, open `docs/spark-DA.pdf`; ask the organiser if it
conflicts with the prepared workflow. See `participants-instructions.md` for the
managed-workshop guide covering both interfaces. Standalone Docker commands are
not setup instructions for this managed JupyterHub workshop.

In [ ]:
spark.stop()
print('Spark session stopped.')
